---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

In [89]:
knitr::opts_chunk$set(echo = TRUE)


## Load Required Libraries

In [90]:
options(verbose = FALSE)
options(warn = -1)


In [91]:
library(here)
source(here("data-cleaning", "r_scripts", "libraries.R"))


In [92]:
options(warn = 1)


## Set Parameters

### Year to Load, Version, and Parameters

In [93]:
source(here("data-cleaning", "r_scripts", "parameters.R"))


## Source Data Formats

In [94]:
source(here("data-cleaning", "r_scripts", "data-formats.R"))
source(here("data-cleaning", "r_scripts", "file-paths.R"))


## Source Functions

In [95]:
source(here("data-cleaning", "r_scripts", "general-functions.R"))
# source(here("data-cleaning", "r_scripts", "clean-data-mini-functions.R"))
source(here("data-cleaning", "r_scripts", "profvis.R"))
source(here("data-cleaning", "r_scripts", "main-functions.R"))
source(here("data-cleaning", "r_scripts", "icd-functions.R"))
source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("data-cleaning", "r_scripts", "grouper-functions.R"))


## Load Mapping Data

In [96]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]
# head(proc)

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]
# head(rvs_icd9)

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))
# head(acr_rvs)

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")
# head(tdrg_icd10)

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read Data

### Reading Data

In [97]:
options(verbose = FALSE)
options(warn = -1)

dt <- main_read_function()

if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
} else {
  total_rows <- fread(full_claims, select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file)
}


[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."


In [98]:
options(warn = 1)


## Data Processing

### Data Cleaning

#### Not chunking

In [99]:
if (!to_chunk) {
  profvis_wrapper <- function(to_profvis, code_block, fname) {
    if (to_profvis) {
      p <- profvis({
        do.call(code_block, list())
      })
      htmlwidgets::saveWidget(
        p,
        file = here("git-ignored-files", "profvis", paste0(fname)),
        selfcontained = TRUE
      )
    } else {
      code_block()
    }
  }

  clean_data <- function() {
    # Add year column
    dt[, SRC_YR := as.integer(year_to_load)]

    # Rename columns
    setnames(dt, old = old_colnames, new = new_colnames)

    # Check if all columns were successfully renamed
    if (!all(new_colnames %in% colnames(dt))) {
      missing_cols <- setdiff(new_colnames, colnames(dt))
      warning("Failed to rename the following columns: ", paste(missing_cols, collapse = ", "))
      stop("Column renaming failed.")
    }

    if (to_view_checks) {
      print("Successfully renamed columns; All expected columns exist")
    }

    # Collapse columns clin_icd1 to clin_icd12 into clin_icd
    dt[, clin_icd := collapse_columns(
      mget(paste0("clin_icd", 1:12),
        envir = as.environment(dt)
      ),
      na_like_strings
    )]
    dt[, paste0("clin_icd", 1:12) := NULL]

    # Collapse columns clin_rvs1 to clin_rvs20 into clin_rvs
    dt[, clin_rvs := collapse_columns(
      mget(paste0("clin_rvs", 1:20),
        envir = as.environment(dt)
      ),
      na_like_strings
    )]
    dt[, paste0("clin_rvs", 1:20) := NULL]

    # Remove lumped ICD codes from clin_icd
    dt[, clin_icd := remove_lumped_icd_codes(dt$clin_icd)]

    # Turn clin_icd and clin_rvs into lists
    dt[, clin_icd := split_to_vector(clin_icd)]
    dt[, clin_rvs := split_to_vector(clin_rvs)]

    # Clean and unlump clin_c1 and clin_c2
    dt[, clin_c1_orig := clin_c1]
    dt[, clin_c1 := clean_column(dt$clin_c1, na_like_strings)]
    clin_c1_cleaning_comparison <- dt[
      clin_c1 != clin_c1_orig,
      .(clin_c1_orig, clin_c1)
    ]
    if (to_view_checks) {
      print(head(clin_c1_cleaning_comparison)) # Check: Print head of changes
    }
    dt[, clin_c1_orig := NULL]

    dt[, clin_c2_orig := clin_c2]
    dt[, clin_c2 := clean_column(dt$clin_c2, na_like_strings)]
    clin_c2_cleaning_comparison <- dt[
      clin_c2 != clin_c2_orig,
      .(clin_c2_orig, clin_c2)
    ]
    if (to_view_checks) {
      print(head(clin_c2_cleaning_comparison)) # Check: Print head of changes
    }
    dt[, clin_c2_orig := NULL]

    dt[, clin_c1 := remove_lumped_icd_codes(dt$clin_c1)]
    dt[, clin_c2 := remove_lumped_icd_codes(dt$clin_c2)]

    dt[, clin_c1 := split_to_vector(clin_c1)]
    clin_c1_result <- transfer_extra_icd10s_to_clin_icd(
      dt$clin_icd, dt$clin_c1
    )
    dt[, clin_icd := clin_c1_result$clin_icd]
    dt[, clin_c1 := clin_c1_result$col_first]

    dt[, clin_c2 := split_to_vector(clin_c2)]
    clin_c2_result <- transfer_extra_icd10s_to_clin_icd(dt$clin_icd, dt$clin_c2)
    dt[, clin_icd := clin_c2_result$clin_icd]
    dt[, clin_c2 := clin_c2_result$col_first]

    clin_c1_rvs_results <- append_and_remove_rvs(
      dt$clin_rvs, dt$clin_c1, rvs_icd9
    )
    dt[, clin_rvs := clin_c1_rvs_results$clin_rvs]
    dt[, clin_c1 := clin_c1_rvs_results$col]

    clin_c2_rvs_results <- append_and_remove_rvs(
      dt$clin_rvs, dt$clin_c2, rvs_icd9
    )
    dt[, clin_rvs := clin_c2_rvs_results$clin_rvs]
    dt[, clin_c2 := clin_c2_rvs_results$col]

    dt[, clin_rvs := lapply(clin_rvs, unique)]
    dedup_result <- ensure_unique_icd_codes(
      dt$clin_c1, dt$clin_c2, dt$clin_icd
    )
    dt[, clin_c1 := dedup_result$clin_c1]
    dt[, clin_c2 := dedup_result$clin_c2]
    dt[, clin_icd := dedup_result$clin_icd]

    # Replace empty strings in character and factor columns with NA
    dt <- replace_empty_with_na(dt, to_view_checks)

    warning_thrown <- FALSE

    # Remap and check for patient type
    result <- remap_patient_type(dt$pat_type)
    dt$pat_type <- result$remapped
    if (length(result$unmapped) > 0 && to_view_checks) {
      warning_thrown <- TRUE
      print("Unmapped Patient Types:")
      print(result$unmapped)
    }
    if (warning_thrown && to_view_checks) {
      print("Patient Types:")
      print(unique(dt$pat_type))
    }

    warning_thrown <- FALSE

    # Remap and check for member category parent
    result <- remap_memcat_parent_desc(dt$pat_memcat_parent)
    dt$pat_memcat_parent <- result$remapped
    if (length(result$unmapped) > 0 && to_view_checks) {
      warning_thrown <- TRUE
      print("Unmapped Memcat Parent Types:")
      print(result$unmapped)
    }
    if (warning_thrown && to_view_checks) {
      print("Memcat Parent Types:")
      print(unique(dt$pat_memcat_parent))
    }

    warning_thrown <- FALSE

    # Remap and check for member category child
    result <- remap_memcat_child_desc(dt$pat_memcat_child)
    dt$pat_memcat_child <- result$remapped
    if (length(result$unmapped) > 0 && to_view_checks) {
      warning_thrown <- TRUE
      print("Unmapped Memcat Child Types:")
      print(result$unmapped)
    }
    if (warning_thrown && to_view_checks) {
      print("Memcat Child Types:")
      print(unique(dt$pat_memcat_child))
    }

    warning_thrown <- FALSE

    # Remap and check for clinical discharge disposition
    result <- remap_disposition(dt$clin_discharge)
    dt$clin_discharge <- result$remapped
    if (length(result$unmapped) > 0 && to_view_checks) {
      warning_thrown <- TRUE
      print("Unmapped Discharge Types:")
      print(result$unmapped)
    }
    if (warning_thrown && to_view_checks) {
      print("Discharge Types:")
      print(unique(dt$clin_discharge))
    }
  }

  profvis_wrapper(to_profvis, clean_data, "clean-data.html")
}


[1] "Successfully renamed columns; All expected columns exist"
   clin_c1_orig clin_c1
         <char>  <char>
1:        A97.1    A971
2:        I21.9    I219
3:        K29.1    K291
4:        J06.9    J069
5:        N39.0    N390
6:        P23.9    P239
   clin_c2_orig clin_c2
         <char>  <char>
1:        E86.1    E861
2:        I63.4    I634
3:        I61.5    I615
4:        E86.1    E861
5:        I63.9    I639
6:        E87.8    E878


Warning message in append_and_remove_rvs(dt$clin_rvs, dt$clin_c1, rvs_icd9):
"Discarded RVS codes: 90375, 77401, 77418, 77261, 76942, 77761, 77421, 47010, 77789, 47000, 77781, 79005, 79000, 55770, 12354, 67340"
Warning message in append_and_remove_rvs(dt$clin_rvs, dt$clin_c2, rvs_icd9):
"Discarded RVS codes: 77418, 77401, 77781, 77761, 77789"




|Column              | "" Replaced| "NA" Replaced| "character(0)" Replaced|
|:-------------------|-----------:|-------------:|-----------------------:|
|time_adm            |           3|             0|                       0|
|time_dis            |           3|             0|                       0|
|date_rec            |           5|             0|                       0|
|date_ref            |       88530|             0|                       0|
|date_check          |        6930|             0|                       0|
|id_hcp              |         115|             0|                       0|
|clin_acc            |        3488|             0|                       0|
|pat_rel             |       61505|             0|                       0|
|pat_bdate           |       21894|             0|                       0|
|pat_memcat_parent   |           6|             0|                       0|
|pat_memcat_child    |           6|             0|                       0|
|pat_memca

In [ ]:
if (!to_chunk) {
  tic("Total execution time:")
  result <- clean_data(dt, to_profvis)
  dt <- result[[1]] # Extract the cleaned data from the result
  if (to_profvis) {
    htmlwidgets::saveWidget(
      result[[2]],
      file = here("git-ignored-files", "profvis", "clean-data.html"),
      selfcontained = TRUE
    )
  }
}


#### Chunking

In [ ]:
if (to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    p <- profvis({
      num_cores <- max(1, availableCores() - 1)
      chunk_size <- ceiling(nrow(dt) / num_cores)
      chunks <- split(dt, rep(1:num_cores,
        each = chunk_size,
        length.out = nrow(dt)
      ))
      # Plan for parallel processing
      plan(multisession, workers = num_cores)
      # Process each chunk in parallel
      processed_chunks <- future_lapply(chunks, process_chunk,
        future.seed = global_seed
      )
      # Combine processed chunks
      dt <- rbindlist(processed_chunks)
      dt <- replace_empty_with_na(dt, to_view_checks)
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "parallelized.html"),
      selfcontained = TRUE
    )
  } else {
    num_cores <- max(1, availableCores() - 1)
    chunk_size <- ceiling(nrow(dt) / num_cores)
    chunks <- split(dt, rep(1:num_cores,
      each = chunk_size,
      length.out = nrow(dt)
    ))
    # Plan for parallel processing
    plan(multisession, workers = num_cores)
    # Process each chunk in parallel
    processed_chunks <- future_lapply(chunks, process_chunk,
      future.seed = global_seed
    )
    # Combine processed chunks
    dt <- rbindlist(processed_chunks)
    dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


### Map codes and Find PDx (if not chunking)

#### Map Codes

In [ ]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
      map_then_compare_icd_mappings(
        tdrg_icd10,
        rows_to_show = 10,
        invalid_rows_to_show = 10
      )

      # Replace empty strings in character and factor columns with NA
      # dt <- replace_empty_with_na(dt, to_view_checks)
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "map_rvs.html"),
      selfcontained = TRUE
    )
  } else {
    dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
    map_then_compare_icd_mappings(
      tdrg_icd10,
      rows_to_show = 10,
      invalid_rows_to_show = 10
    )

    # Replace empty strings in character and factor columns with NA
    # dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


#### Find PDx

In [ ]:
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      # dt <- apply_find_pdx(dt)
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "find_pdx.html"),
      selfcontained = TRUE
    )
  } else {
    # dt <- apply_find_pdx(dt)
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}


In [ ]:
if (to_write) {
  fwrite(dt, here(path_to_intermediate, paste0(
    "output_", year_to_load,
    suffix, ".csv"
  )))
}


## Export

### Export for Batch Grouper

In [ ]:
if (to_group) {
  if (to_profvis) {
    p <- profvis({
      export_for_batch_grouper(dt, year_to_load, output_txt_file)
      for_batch_grouping <- fread(output_txt_file,
        sep = "|", na.strings = "--"
      )
      batch_grouping_result <- fread(grouper_result_file,
        sep = "|", na.strings = "--"
      )
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "export_for_grouper.html"),
      selfcontained = TRUE
    )
  } else {
    export_for_batch_grouper(dt, year_to_load, output_txt_file)
    for_batch_grouping <- fread(output_txt_file,
      sep = "|", na.strings = "--"
    )
    batch_grouping_result <- fread(grouper_result_file,
      sep = "|", na.strings = "--"
    )
  }
}


## Runtime Estimation

### Stop Timer

In [ ]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


### Calculate Speed

In [ ]:
# Calculate time spent per cell and per row
total_rows_dt <- nrow(dt)
total_cells <- nrow(dt) * ncol(dt)

time_per_cell <- total_time / total_cells
time_per_row <- total_time / total_rows_dt
time_estimate_total_rows <- time_per_row * total_rows

# Format the row numbers
formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
formatted_total_rows <- format_large_numbers(total_rows)

# Print the results with aligned decimal points and formatted row numbers
cat(sprintf(
  "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
  formatted_total_rows_dt, total_time
))
cat(sprintf(
  "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
  formatted_total_rows_dt, time_per_row * 1000
))
cat(sprintf(
  "Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
  formatted_total_rows, time_estimate_total_rows / 60
))
